In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    print(root, len(files))


    

import torch
print("device:", "cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import torch
print("device:", "cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import os, random, json
import numpy as np, pandas as pd
import torch, torch.nn as nn
from PIL import Image, ImageDraw
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
SPLIT_PATH = "/kaggle/input/datasets/ayeshasiddikaahona/neuroskin/split.csv"
df = pd.read_csv(SPLIT_PATH)

CLASSES = sorted(df["label"].unique())
print(CLASSES)
class_to_idx = {c: i for i, c in enumerate(CLASSES)}
print(df["split"].value_counts().to_dict())
print("first file exists:", os.path.exists(df.path.iloc[0]))

IMG = 224

def hide_corners(img):
    w, h = img.size
    d = ImageDraw.Draw(img)
    d.rectangle([0, int(h * 0.80), int(w * 0.30), h], fill=(0, 0, 0))
    d.rectangle([int(w * 0.70), int(h * 0.80), w, h], fill=(0, 0, 0))
    return img

norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(), norm])
eval_tf = transforms.Compose([transforms.ToTensor(), norm])

class SkinData(Dataset):
    def __init__(self, frame, tf):
        self.frame = frame.reset_index(drop=True)
        self.tf = tf
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, i):
        row = self.frame.iloc[i]
        img = Image.open(row.path).convert("RGB").resize((IMG, IMG))
        img = hide_corners(img)
        return self.tf(img), class_to_idx[row.label]

train_ds = SkinData(df[df.split == "train"], train_tf)
val_ds   = SkinData(df[df.split == "val"],   eval_tf)

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
val_dl   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2)
print("train:", len(train_ds), "| val:", len(val_ds))

In [ ]:
model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
in_f = model.classifier[3].in_features
model.classifier[3] = nn.Linear(in_f, len(CLASSES))
model = model.to(device)

loss_fn = nn.CrossEntropyLoss()

def run_epoch(loader, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = loss_fn(out, y)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(y)
            correct += (out.argmax(1) == y).sum().item()
            n += len(y)
    return total_loss / n, correct / n

history = []
best_val = 0.0

def fit(epochs, lr, tag):
    global best_val
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    for e in range(1, epochs + 1):
        tl, ta = run_epoch(train_dl, opt)
        vl, va = run_epoch(val_dl)
        history.append({"stage": tag, "epoch": e, "train_loss": tl, "train_acc": ta,
                        "val_loss": vl, "val_acc": va})
        print(f"{tag} epoch {e}: train acc {ta:.3f} | val acc {va:.3f} | val loss {vl:.3f}")
        if va > best_val:
            best_val = va
            torch.save(model.state_dict(), "/kaggle/working/skin_model.pth")
            print("   saved best model")

# Step 1: only the new last layer
for p in model.features.parameters():
    p.requires_grad = False
fit(3, 1e-3, "head")

# Step 2: fine-tune everything with a small learning rate
for p in model.parameters():
    p.requires_grad = True
fit(5, 1e-4, "finetune")

print("best val acc:", best_val)

In [ ]:
pd.DataFrame(history).to_csv("/kaggle/working/training_log.csv", index=False)
with open("/kaggle/working/classes.json", "w") as f:
    json.dump(CLASSES, f)
print(os.listdir("/kaggle/working"))